# 59P1 V8.2 Observed-Print Deployment Fit and Freeze

**GPU runtime.** This notebook refits the selected V8.2 33×33 `TopologyCNN` on all permitted stable development rows, fits one final isotonic calibrator from preserved source-group OOF raw scores, writes an immutable runtime bundle for 60R2, and stores replay evidence in the sibling `replay/` directory.

It makes no new architecture, threshold, or promotion decision.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision joblib

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'; os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,random,sys,platform,shutil,time,zipfile
import cv2,torch,joblib,numpy as np,pandas as pd
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.isotonic import IsotonicRegression
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT=ROOT/'observed_print_classifier_v8_2'; OUT_ROOT=ROOT/'observed_print_deployment_freeze_v8_2'
SOURCE_V82_RUN_OVERRIDE=None
SEED=42; PATCH=33; HALF=16; BATCH=512; EPOCHS=12; LR=1e-3; WD=1e-4
PRINT_T=.995; ANNOTATION_T=.995; PRINT_MARGIN=.990; MIN_RUN=2; DARK_T=200
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Select a GPU runtime.')
torch.use_deterministic_algorithms(True); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
def seed_all(): random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
def req(x,m):
 if not x: raise RuntimeError(m)
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def oh(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def ah(a):
 a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def writej(p,x): p.write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook59p1_v82_observed_print_freeze'
OUT=OUT_ROOT/RUN; BUNDLE=OUT/'bundle'; REPLAY=OUT/'replay'; AUDIT=OUT/'audit'; TABLES=OUT/'tables'; UPLOAD=OUT/'upload'
for p in [BUNDLE,REPLAY,AUDIT,TABLES,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('GPU:',torch.cuda.get_device_name(0)); print('Output:',OUT)

In [ ]:
REQUIRED=['experiment_identity.json','development_summary.json','inputs/development_score_manifest.csv','inputs/fold_map.csv','inputs/channel_schema.json','tables/stable_sample_manifest.csv','tables/stable_raw_oof.npy','tables/stable_calibrated_oof.npy','audit/model_inventory.csv','audit/calibrator_inventory.csv']
def complete(root): return sorted([p for p in root.iterdir() if p.is_dir() and all((p/x).exists() for x in REQUIRED)],key=lambda p:p.stat().st_mtime) if root.exists() else []
if SOURCE_V82_RUN_OVERRIDE:
 SOURCE=Path(SOURCE_V82_RUN_OVERRIDE); req(all((SOURCE/x).exists() for x in REQUIRED),'Source override incomplete.')
else:
 hits=complete(V82_ROOT); req(hits,f'No complete V8.2 run below {V82_ROOT}'); SOURCE=hits[-1]
identity=json.loads((SOURCE/'experiment_identity.json').read_text()); schema0=json.loads((SOURCE/'inputs/channel_schema.json').read_text())
req(identity.get('version')=='8.2-development-whole-page','Wrong source version.')
req(identity.get('independent_data_used') is False,'Source used independent data.')
for k,v in [('seed',SEED),('patch',PATCH),('epochs',EPOCHS)]: req(int(identity[k])==v,f'{k} mismatch')
req(identity['channels']==CHANNELS and schema0['channels']==CHANNELS,'Channel mismatch.')
req(float(identity['lr'])==LR and float(identity['weight_decay'])==WD,'Training hyperparameter mismatch.')
req(float(identity['print_threshold'])==PRINT_T and float(identity['annotation_threshold'])==ANNOTATION_T and float(identity['print_margin'])==PRINT_MARGIN,'Policy mismatch.')
stable=pd.read_csv(SOURCE/'tables/stable_sample_manifest.csv'); raw_oof=np.load(SOURCE/'tables/stable_raw_oof.npy').astype(np.float64); calibrated_oof=np.load(SOURCE/'tables/stable_calibrated_oof.npy').astype(np.float64)
req({'page','source_group','y','x','truth','fold'}.issubset(stable.columns),'Stable schema incomplete.')
req(len(stable)==len(raw_oof)==len(calibrated_oof)==int(identity['stable_rows']),'Stable lengths differ.')
req(stable.truth.isin([0,1]).all() and not stable.duplicated(['page','y','x']).any(),'Stable roster invalid.')
req(np.isfinite(raw_oof).all() and np.isfinite(calibrated_oof).all(),'OOF scores non-finite.')
verified=[]
for role,key,pathkey,hashkey,folder in [('model','models','checkpoint','checkpoint_sha256','models'),('calibrator','calibrators','joblib','joblib_sha256','calibrators')]:
 for item in identity[key]:
  p=Path(item[pathkey]); p=p if p.exists() else SOURCE/folder/p.name
  req(p.exists() and sha(p)==item[hashkey],f'{role} verification failed: {p.name}')
  verified.append({'role':role,'path':str(p),'sha256':sha(p)})
pd.DataFrame(verified).to_csv(AUDIT/'source_v82_object_verification.csv',index=False)
print('Source:',SOURCE,'Rows:',len(stable),'Pages:',stable.page.nunique())

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
V75=Path(identity['v7_5_reference']); status=pd.read_csv(V75/'tables/cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}; observed=imap(ROOT/'aligned')
def paper(obs,mask):
 vis=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),45); den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),45); fallback=cv2.GaussianBlur(obs,(0,0),45); return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
runtime={}; audit=[]; pages=sorted(stable.page.astype(str).unique()); keys=['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page in pages:
 req(page in cache and cache[page].exists() and page in observed,f'{page}: missing source arrays')
 with np.load(cache[page]) as z:
  req(all(k in z.files for k in keys),f'{page}: cache incomplete'); a={k:z[k].copy() for k in keys}
 obs=cv2.imread(str(observed[page]),0); req(obs is not None and all(v.shape==obs.shape for v in a.values()),f'{page}: shape failure')
 mask=a['mask']>0; bg=paper(obs,mask); norm=cv2.divide(obs,np.maximum(bg,1),scale=255)
 runtime[page]={'obs':obs,'norm':norm,'mask':mask,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,'edge':cv2.Canny(obs,70,170)>0}
 audit.append({'page':page,'observed':str(observed[page]),'observed_sha256':sha(observed[page]),'cache':str(cache[page]),'cache_sha256':sha(cache[page])})
pd.DataFrame(audit).to_csv(AUDIT/'input_identity.csv',index=False)
padded={p:{k:np.pad((z[k].astype(np.float32)/255 if k in ['obs','norm'] else z[k].astype(np.float32)),HALF,mode='reflect') for k in CHANNELS} for p,z in runtime.items()}
page_index={p:i for i,p in enumerate(pages)}; pid=stable.page.astype(str).map(page_index).to_numpy(np.int16); yy=stable.y.to_numpy(np.int32); xx=stable.x.to_numpy(np.int32); truth=stable.truth.to_numpy(np.int8)
def patch(i):
 p=pages[int(pid[i])]; return np.stack([padded[p][k][yy[i]:yy[i]+PATCH,xx[i]:xx[i]+PATCH] for k in CHANNELS]).astype(np.float32)
class DS(Dataset):
 def __init__(self,idx): self.idx=np.asarray(idx)
 def __len__(self): return len(self.idx)
 def __getitem__(self,i):
  j=int(self.idx[i]); return torch.from_numpy(patch(j)),torch.tensor(int(truth[j]),dtype=torch.long)
def wi(w): random.seed(SEED+w); np.random.seed(SEED+w)
def loader(idx,shuffle):
 g=torch.Generator(); g.manual_seed(SEED); return DataLoader(DS(idx),batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,generator=g,worker_init_fn=wi)
class TopologyCNN(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.Conv2d(11,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)
 def forward(self,x): return self.head(self.net(x).flatten(1))
def predict(model,idx):
 out=[]; model.eval()
 with torch.no_grad():
  for xb,_ in loader(idx,False): out.extend(torch.softmax(model(xb.to(DEVICE,non_blocking=True)),1)[:,1].cpu().numpy())
 return np.asarray(out,np.float64)
print('Exact tensor builder ready.')

In [ ]:
seed_all(); idx=np.arange(len(stable)); model=TopologyCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD); counts=np.bincount(truth,minlength=2); weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE); lossfn=nn.CrossEntropyLoss(weight=weights); hist=[]; started=time.time()
for ep in range(EPOCHS):
 model.train(); total=0; seen=0
 for xb,yb in loader(idx,True):
  xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=lossfn(model(xb),yb); loss.backward(); opt.step(); total+=float(loss.item())*len(yb); seen+=len(yb)
 hist.append({'epoch':ep+1,'mean_loss':total/seen,'rows':seen}); print(ep+1,hist[-1]['mean_loss'])
train_seconds=time.time()-started; pd.DataFrame(hist).to_csv(TABLES/'training_history.csv',index=False)
MODEL=BUNDLE/'observed_print_model.pt'; torch.save(model.state_dict(),MODEL)
cal=IsotonicRegression(out_of_bounds='clip').fit(raw_oof,truth); CAL=BUNDLE/'observed_print_calibrator.joblib'; joblib.dump(cal,CAL)
cal_record={'method':'IsotonicRegression','source':'preserved V8.2 source-group OOF raw scores and truth','rows':len(raw_oof),'raw_oof_sha256':sha(SOURCE/'tables/stable_raw_oof.npy'),'stable_manifest_sha256':sha(SOURCE/'tables/stable_sample_manifest.csv'),'x_thresholds':cal.X_thresholds_.tolist(),'y_thresholds':cal.y_thresholds_.tolist()}; cal_record['semantic_sha256']=oh({'x':cal_record['x_thresholds'],'y':cal_record['y_thresholds']}); writej(AUDIT/'calibration_fitting_record.json',cal_record)
req(np.array_equal(joblib.load(CAL).predict(raw_oof),cal.predict(raw_oof)),'Calibrator reload mismatch.')
print('Model:',sha(MODEL)); print('Calibrator:',sha(CAL))

In [ ]:
SCHEMA=BUNDLE/'channel_schema.json'; POLICY=BUNDLE/'policy.json'; CONTRACT=BUNDLE/'runtime_contract.json'; PRE=BUNDLE/'preprocessing_identity.json'; ENV=BUNDLE/'software_environment.json'
channel={'schema_version':'v8.2-observed-print-11-channel-33x33','patch':33,'half_patch':16,'ordered_channels':CHANNELS,'channels':CHANNELS,'dtype':'float32','padding':'reflect','normalisation':{'obs':'divide_255','norm':'divide_255','other':'native_float32'}}; writej(SCHEMA,channel)
compat=oh({'model':sha(MODEL),'calibrator':sha(CAL),'schema':sha(SCHEMA),'print_threshold':PRINT_T,'annotation_threshold':ANNOTATION_T,'print_margin':PRINT_MARGIN,'connectivity':8,'min_run':MIN_RUN}); policy_id='V8_2_OBSERVED_PRINT_DEPLOYMENT_'+compat[:16].upper()
policy={'policy_id':policy_id,'compatibility_id':compat,'model_file':MODEL.name,'model_sha256':sha(MODEL),'calibrator_file':CAL.name,'calibrator_sha256':sha(CAL),'channel_schema_file':SCHEMA.name,'channel_schema_sha256':sha(SCHEMA),'print_threshold':PRINT_T,'annotation_threshold':ANNOTATION_T,'print_margin':PRINT_MARGIN,'effective_binary_print_threshold':.995,'connectivity':8,'minimum_print_run':MIN_RUN,'dark_threshold':DARK_T,'mask_authority':'human-reviewed authoritative mask','yolo_confidence_is_p_print':False,'production_authority':False}; writej(POLICY,policy)
writej(CONTRACT,{'policy_id':policy_id,'compatibility_id':compat,'required_runtime_inputs':['aligned observed page','human-reviewed corrected mask','exact V7.8 observable channels'],'eligible_population':'inside prepared mask, observed < 200, not already preserve','output':'calibrated p_print and decisions','outside_mask_immutable':True,'notebook06_confidence_permitted_as_substitute':False})
writej(PRE,{'policy_id':policy_id,'source_v82_run':str(SOURCE),'source_identity_sha256':sha(SOURCE/'experiment_identity.json'),'tensor_builder':'reflect pad 16; ordered 33x33 patches','architecture':'TopologyCNN-v8.2','seed':SEED,'epochs':EPOCHS,'batch':BATCH,'lr':LR,'weight_decay':WD,'training_rows':len(stable),'pages':stable.page.nunique(),'source_groups':stable.source_group.nunique()})
writej(ENV,{'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,'joblib':joblib.__version__,'platform':platform.platform()})
print(policy_id,compat)

In [ ]:
# Exact reload replay on an ordered multi-page sample.
parts=[]
for p in pages[:16]: parts.extend(stable.index[stable.page.astype(str)==p].to_numpy()[:256].tolist())
ridx=np.asarray(sorted(set(parts))[:4096],dtype=np.int64); req(len(ridx)>0 and stable.iloc[ridx].page.nunique()>=5,'Replay sample inadequate.')
raw1=predict(model,ridx); p1=cal.predict(raw1)
model2=TopologyCNN().to(DEVICE); model2.load_state_dict(torch.load(MODEL,map_location=DEVICE),strict=True); cal2=joblib.load(CAL); raw2=predict(model2,ridx); p2=cal2.predict(raw2)
req(np.array_equal(raw1,raw2) and np.array_equal(p1,p2),'Exact model/calibrator replay failed.')
def decide(p):
 q=1-p; d=np.full(len(p),'UNCERTAIN',object); d[q>=ANNOTATION_T]='ANNOTATION'; d[(p>=PRINT_T)&((p-q)>=PRINT_MARGIN)]='PRINT'; return d
d1=decide(p1); d2=decide(p2); req(np.array_equal(d1,d2),'Decision replay failed.')
sample=stable.iloc[ridx][['page','source_group','y','x','truth']].copy(); sample['raw_print']=raw2; sample['p_print']=p2; sample['decision_pre_run_rule']=d2; sample.to_parquet(REPLAY/'replay_sample_predictions.parquet',index=False); np.savez_compressed(REPLAY/'replay_expected_arrays.npz',indices=ridx,raw_print=raw2,p_print=p2,decision=np.asarray(d2,dtype='U16'))
replay={'passed':True,'policy_id':policy_id,'compatibility_id':compat,'rows':len(ridx),'pages':sample.page.nunique(),'source_groups':sample.source_group.nunique(),'model_reload_exact':True,'calibrator_reload_exact':True,'decision_reload_exact':True,'raw_array_sha256':ah(raw2),'p_print_array_sha256':ah(p2),'decision_array_sha256':ah(np.asarray(d2,dtype='U16')),'sample_sha256':sha(REPLAY/'replay_sample_predictions.parquet'),'arrays_sha256':sha(REPLAY/'replay_expected_arrays.npz')}; writej(REPLAY/'replay_test.json',replay); print(json.dumps(replay,indent=2))

In [ ]:
FREEZE=BUNDLE/'freeze_manifest.json'; objects=[]
for p in sorted(BUNDLE.iterdir()):
 if p.is_file() and p.name!=FREEZE.name: objects.append({'relative_path':p.name,'bytes':p.stat().st_size,'sha256':sha(p)})
freeze={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'59P1-v8.2-observed-print-deployment-freeze','deployment_bundle_complete':True,'policy_id':policy_id,'compatibility_id':compat,'source_v82_run':str(SOURCE),'source_v82_identity_sha256':sha(SOURCE/'experiment_identity.json'),'source_stable_manifest_sha256':sha(SOURCE/'tables/stable_sample_manifest.csv'),'source_raw_oof_sha256':sha(SOURCE/'tables/stable_raw_oof.npy'),'training_rows':len(stable),'training_pages':stable.page.nunique(),'training_source_groups':stable.source_group.nunique(),'calibration_source':'preserved V8.2 source-group OOF raw scores and truth','replay_file':'../replay/replay_test.json','replay_sha256':sha(REPLAY/'replay_test.json'),'objects':objects,'production_authority':False}; writej(FREEZE,freeze)
for x in objects: req((BUNDLE/x['relative_path']).exists() and sha(BUNDLE/x['relative_path'])==x['sha256'],f"Hash failure: {x['relative_path']}")
fit={'source_v82_run':str(SOURCE),'selected_architecture':'TopologyCNN-v8.2','new_model_selection':False,'all_development_refit':True,'training_rows':len(stable),'pages':stable.page.nunique(),'source_groups':stable.source_group.nunique(),'class_annotation':int((truth==0).sum()),'class_print':int((truth==1).sum()),'training_seconds':train_seconds,'model_sha256':sha(MODEL),'calibrator_sha256':sha(CAL),'policy_id':policy_id,'compatibility_id':compat}; writej(AUDIT/'all_development_fit_record.json',fit)
print('Freeze integrity: PASS')

In [ ]:
# Curated review ZIP. Binary runtime objects remain in the durable bundle.
readme='\n'.join(['59P1 V8.2 observed-print deployment freeze','',f'Run: {OUT}',f'Policy ID: {policy_id}',f'Compatibility ID: {compat}','','Runtime requires bundle/observed_print_model.pt, bundle/observed_print_calibrator.joblib and replay/replay_test.json.'])
(UPLOAD/'README.txt').write_text(readme)
items=[SCHEMA,POLICY,CONTRACT,PRE,ENV,FREEZE,REPLAY/'replay_test.json',AUDIT/'all_development_fit_record.json',AUDIT/'calibration_fitting_record.json',AUDIT/'source_v82_object_verification.csv',AUDIT/'input_identity.csv',TABLES/'training_history.csv']; rows=[]
for src in items:
 sub='bundle' if src.parent==BUNDLE else ('replay' if src.parent==REPLAY else ('audit' if src.parent==AUDIT else 'tables')); dst=UPLOAD/sub/src.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha(dst)})
rows += [{'relative_path':'[DURABLE_ONLY]/bundle/'+MODEL.name,'bytes':MODEL.stat().st_size,'sha256':sha(MODEL)},{'relative_path':'[DURABLE_ONLY]/bundle/'+CAL.name,'bytes':CAL.stat().st_size,'sha256':sha(CAL)}]
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in rows))
ZIP=UPLOAD/'59P1_V8_2_Observed_Print_Deployment_Fit_and_Freeze_upload_bundle.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(UPLOAD.rglob('*')):
  if p.is_file() and p!=ZIP: z.write(p,arcname=str(p.relative_to(UPLOAD)))
req(ZIP.exists() and ZIP.stat().st_size>0,'Upload ZIP failed.')
required=['observed_print_model.pt','observed_print_calibrator.joblib','channel_schema.json','policy.json','runtime_contract.json','preprocessing_identity.json','software_environment.json','freeze_manifest.json']; req(all((BUNDLE/x).exists() for x in required),'Bundle incomplete.'); req((REPLAY/'replay_test.json').exists(),'Sibling replay file missing.')
verdict={'verdict':'PASS_OBSERVED_PRINT_DEPLOYMENT_BUNDLE_READY_FOR_60R2','run':str(OUT),'bundle':str(BUNDLE),'replay':str(REPLAY/'replay_test.json'),'policy_id':policy_id,'compatibility_id':compat,'model_sha256':sha(MODEL),'calibrator_sha256':sha(CAL),'freeze_sha256':sha(FREEZE),'replay_sha256':sha(REPLAY/'replay_test.json'),'production_authority':False}; writej(OUT/'notebook59p1_verdict.json',verdict); (OUT/'executive_summary.txt').write_text(json.dumps(verdict,indent=2)); print(json.dumps(verdict,indent=2)); print('Upload:',ZIP)